In [1]:
!pip install langchain langgraph 

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install dotenv

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
!pip install langchain-nvidia-ai-endpoints langchain-core langchain-community

In [5]:
from langgraph.graph import StateGraph,START,END,add_messages
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from typing import TypedDict,Annotated,List,Literal,operator
from langchain_core.messages import BaseMessage,HumanMessage,SystemMessage,AIMessage
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode,tools_condition
from pathlib import Path
import string
import os
import shutil

C:\Users\kiran\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
@tool
def cpu_monitor() -> dict:
    """
    Returns live CPU information for AI analysis.
    """
    import psutil

    # Prime CPU counters
    psutil.cpu_percent(interval=None, percpu=True)

    overall = psutil.cpu_percent(interval=1)
    per_core = psutil.cpu_percent(interval=None, percpu=True)
    freq = psutil.cpu_freq()

    temperature = None
    power_watts = None

    # Optional: LibreHardwareMonitor
    try:
        import clr

        clr.AddReference("LibreHardwareMonitorLib")
        from LibreHardwareMonitor.Hardware import Computer

        computer = Computer()
        computer.IsCpuEnabled = True
        computer.Open()

        for hw in computer.Hardware:
            hw.Update()

            for sensor in hw.Sensors:

                if sensor.SensorType.ToString() == "Temperature":
                    if "CPU Package" in sensor.Name or "CPU" in sensor.Name:
                        temperature = round(sensor.Value, 1)

                elif sensor.SensorType.ToString() == "Power":
                    if "CPU Package" in sensor.Name or "CPU" in sensor.Name:
                        power_watts = round(sensor.Value, 2)

    except Exception:
        pass

    # Prime process cpu counters
    for p in psutil.process_iter():
        try:
            p.cpu_percent(None)
        except Exception:
            pass

    psutil.cpu_percent(interval=0.5)

    processes = []

    for p in psutil.process_iter(
        ["pid", "name", "cpu_percent", "num_threads"]
    ):
        try:
            cpu = p.cpu_percent(None)

            if cpu > 0:
                processes.append(
                    {
                        "pid": p.info["pid"],
                        "name": p.info["name"],
                        "cpu_percent": round(cpu, 2),
                        "threads": p.info["num_threads"],
                    }
                )

        except Exception:
            continue

    processes.sort(
        key=lambda x: x["cpu_percent"],
        reverse=True,
    )

    stats = psutil.cpu_stats()

    return {
        "usage_percent": overall,

        "per_core_usage": per_core,

        "physical_cores": psutil.cpu_count(logical=False),

        "logical_cores": psutil.cpu_count(logical=True),

        "frequency": {
            "current_mhz": round(freq.current, 2) if freq else None,
            "max_mhz": round(freq.max, 2) if freq else None,
        },

        "temperature_c": temperature,

        "power_watts": power_watts,

        "context_switches": stats.ctx_switches,

        "interrupts": stats.interrupts,

        "top_processes": processes[:10],
    }

# {
#   "usage_percent": 42.1,
#   "per_core_usage": [35.2, 47.1, 39.8, 46.0],
#   "physical_cores": 8,
#   "logical_cores": 16,
#   "frequency": {
#     "current_mhz": 4188,
#     "max_mhz": 4700
#   },
#   "temperature_c": 61.5,
#   "power_watts": 28.3,
#   "context_switches": 12544534,
#   "interrupts": 9234112,
#   "top_processes": [
#     {
#       "pid": 10244,
#       "name": "chrome.exe",
#       "cpu_percent": 34.8,
#       "threads": 58
#     }
#   ]
# }

@tool
def ram_monitor() -> dict:
    """
    Returns live RAM information for AI analysis.
    """
    import psutil

    vm = psutil.virtual_memory()
    swap = psutil.swap_memory()

    processes = []

    for p in psutil.process_iter(
        [
            "pid",
            "name",
            "memory_info",
            "memory_percent",
            "status",
        ]
    ):
        try:
            mem = p.info["memory_info"].rss / (1024 * 1024)

            processes.append(
                {
                    "pid": p.info["pid"],
                    "name": p.info["name"],
                    "memory_mb": round(mem, 2),
                    "memory_percent": round(
                        p.info["memory_percent"], 2
                    ),
                    "status": p.info["status"],
                }
            )

        except Exception:
            continue

    processes.sort(
        key=lambda x: x["memory_mb"],
        reverse=True,
    )

    return {
        "total_gb": round(vm.total / (1024**3), 2),

        "used_gb": round(vm.used / (1024**3), 2),

        "available_gb": round(vm.available / (1024**3), 2),

        "free_gb": round(vm.free / (1024**3), 2),

        "usage_percent": vm.percent,

        "cached_gb": round(
            getattr(vm, "cached", 0) / (1024**3),
            2,
        ),

        "pagefile_used_gb": round(
            swap.used / (1024**3),
            2,
        ),

        "pagefile_total_gb": round(
            swap.total / (1024**3),
            2,
        ),

        "top_memory_processes": processes[:10],
    }


# {
#     "total_gb": 16.0,
#     "used_gb": 11.7,
#     "available_gb": 4.3,
#     "free_gb": 3.8,
#     "usage_percent": 73.4,
#     "cached_gb": 1.2,
#     "pagefile_used_gb": 0.8,
#     "pagefile_total_gb": 2.0,
#     "top_memory_processes": [
#         {
#             "pid": 3212,
#             "name": "chrome.exe",
#             "memory_mb": 2145.63,
#             "memory_percent": 13.08,
#             "status": "running"
#         },
#         {
#             "pid": 4876,
#             "name": "code.exe",
#             "memory_mb": 1120.55,
#             "memory_percent": 6.84,
#             "status": "running"
#         }
#     ]
# }




@tool
def process_monitor() -> list:
    """
    Returns important information about running processes.
    """
    import psutil
    import subprocess
    from datetime import datetime

    processes = []

    priority_map = {
        getattr(psutil, "IDLE_PRIORITY_CLASS", -1): "Idle",
        getattr(psutil, "BELOW_NORMAL_PRIORITY_CLASS", -1): "Below Normal",
        getattr(psutil, "NORMAL_PRIORITY_CLASS", -1): "Normal",
        getattr(psutil, "ABOVE_NORMAL_PRIORITY_CLASS", -1): "Above Normal",
        getattr(psutil, "HIGH_PRIORITY_CLASS", -1): "High",
        getattr(psutil, "REALTIME_PRIORITY_CLASS", -1): "Realtime",
    }

    # Prime CPU counters
    for p in psutil.process_iter():
        try:
            p.cpu_percent(None)
        except Exception:
            pass

    psutil.cpu_percent(interval=0.5)

    for p in psutil.process_iter(
        [
            "pid",
            "name",
            "exe",
            "status",
            "create_time",
            "memory_info",
            "num_threads",
        ]
    ):
        try:
            cpu = p.cpu_percent(None)
            mem = round(p.info["memory_info"].rss / (1024 * 1024), 2)

            try:
                io = p.io_counters()
                read_mb = round(io.read_bytes / (1024 * 1024), 2)
                write_mb = round(io.write_bytes / (1024 * 1024), 2)
            except Exception:
                read_mb = 0
                write_mb = 0

            try:
                connections = len(p.net_connections())
            except Exception:
                connections = 0

            try:
                priority = priority_map.get(
                    p.nice(),
                    str(p.nice())
                )
            except Exception:
                priority = "Unknown"

            runtime = round(
                (datetime.now().timestamp() - p.info["create_time"]) / 60,
                1,
            )

            # Company Name (Windows)
            company = None
            try:
                if p.info["exe"]:
                    cmd = [
                        "powershell",
                        "-NoProfile",
                        "-Command",
                        f"(Get-Item '{p.info['exe']}').VersionInfo.CompanyName"
                    ]
                    company = subprocess.run(
                        cmd,
                        capture_output=True,
                        text=True,
                        timeout=2
                    ).stdout.strip()
            except Exception:
                pass

            processes.append(
                {
                    "pid": p.info["pid"],
                    "name": p.info["name"],
                    "cpu_percent": round(cpu, 2),
                    "memory_mb": mem,
                    "status": p.info["status"],
                    "threads": p.info["num_threads"],
                    "priority": priority,
                    "exe": p.info["exe"],
                    "company": company,
                    "network_connections": connections,
                    "disk_read_mb": read_mb,
                    "disk_write_mb": write_mb,
                    "running_time_minutes": runtime,
                }
            )

        except Exception:
            continue

    processes.sort(
        key=lambda x: (
            x["cpu_percent"],
            x["memory_mb"]
        ),
        reverse=True,
    )

    return processes


# [
#   {
#     "pid": 5432,
#     "name": "chrome.exe",
#     "cpu_percent": 32.4,
#     "memory_mb": 1850.6,
#     "status": "running",
#     "threads": 54,
#     "priority": "Normal",
#     "exe": "C:\\Program Files\\Google\\Chrome\\Application\\chrome.exe",
#     "company": "Google LLC",
#     "network_connections": 23,
#     "disk_read_mb": 145.3,
#     "disk_write_mb": 22.7,
#     "running_time_minutes": 184.2
#   }
# ]